# PXDesign on Colab

> **Version 1.2.1.** The current version is always the one at
> [the releases page](https://github.com/a-LittleMonk/PXDesign/releases).
> If that page shows a higher number than the line you are reading, this copy
> is out of date — [open the current notebook](https://colab.research.google.com/github/a-LittleMonk/PXDesign/blob/main/PXDesign_colab.ipynb)
> and *File > Save a copy in Drive* to replace it. A saved copy never updates itself.

The whole PXDesign pipeline: **design** binders against a target you pick, with
hotspots, using the PXDesign-d diffusion generator; then **evaluate** them — put
sequences on the backbones with ProteinMPNN, refold those sequences with
AlphaFold2 / Protenix / ESMFold, and score how far the refold drifts from the
design. Run steps 1-3 once (~25 min), then 4-6 as often as you like.

Either half works alone: skip 4 and 5 to score designs you already have from
RFdiffusion or BindCraft, or stop after 5 if you only want backbones.

**First: Runtime -> Change runtime type -> T4 GPU.**

<small>**Using Run all?** Two things need you: step 2 waits for a Google Drive
authorisation popup, and step 6's fields must already say what you want evaluated —
its defaults reproduce `binder_eval_demo.sh` on the five bundled PD-L1 binder
examples and ignore anything in Drive. Set those two up and the rest, including
saving back to Drive, runs unattended. The setup steps raise on a bad environment,
so a Run all stops at the real cause instead of cascading.</small>

<small>Runs [PXDesign](https://github.com/bytedance/PXDesign) and
[PXDesignBench](https://github.com/bytedance/PXDesignBench) unmodified, inside one
Python 3.11 environment, called as programs rather than imported, because Colab's
own Python is too new for the pinned PyTorch / JAX this stack needs. Every piece of
design, prediction and scoring logic is upstream's — this notebook only builds the
environment, fetches weights, and hands the `pxdesign` CLI and `pxdbench/run.py` the
same arguments upstream's own README and demo scripts do. Double-click a step title
to see its code.</small>

In [ ]:
#@title 1. Check the GPU
#@markdown Must show a GPU. If it does not, fix the runtime type before going
#@markdown further: *Runtime > Change runtime type > T4 GPU*.
#@markdown
#@markdown Also reports whether the card can do bfloat16, which decides the
#@markdown Protenix dtype steps 5 and 6 should use.

NOTEBOOK_VERSION = "1.2.1"
print(f"PXDesign on Colab - notebook version {NOTEBOOK_VERSION}\n")

!nvidia-smi || echo "nvidia-smi found no GPU"

try:
    import torch
    assert torch.cuda.is_available()
except Exception as exc:
    raise RuntimeError(
        "No GPU visible. Runtime > Change runtime type > T4 GPU, then run this "
        "cell again. Nothing below works on CPU."
    ) from exc

_name = torch.cuda.get_device_name(0)
_major, _minor = torch.cuda.get_device_capability(0)
_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16_OK = _major >= 8

print(f"\nGPU         : {_name}")
print(f"compute     : {_major}.{_minor}")
print(f"memory      : {_mem:.1f} GB")
print(f"bfloat16    : {'yes' if BF16_OK else 'no - keep dtype at fp32 in steps 5 and 6'}")
if _mem < 14:
    print("\nUnder 14 GB of VRAM. Protenix on a long target may run out of memory;")
    print("evaluate fewer sequences at a time, or turn the Protenix filter off.")

In [ ]:
#@title 2. Mount your Google Drive
#@markdown Gives the notebook a folder on your Drive, so you can feed it designs
#@markdown you keep there, have the results land back there, and keep the model
#@markdown weights between sessions instead of downloading them again.
#@markdown A browser popup will ask you to authorise access.
drive_folder = "PXDesign" #@param {type:"string"}

import os
from datetime import datetime
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT    = f"/content/drive/MyDrive/{drive_folder}"
DRIVE_INPUTS  = f"{DRIVE_ROOT}/inputs"
DRIVE_OUTPUT  = f"{DRIVE_ROOT}/outputs"
DRIVE_WEIGHTS = f"{DRIVE_ROOT}/tool_weights"
for _d in (DRIVE_INPUTS, DRIVE_OUTPUT, DRIVE_WEIGHTS):
    os.makedirs(_d, exist_ok=True)

with open("/content/.pxd_drive", "w") as f:
    f.write(DRIVE_ROOT)

print(f"Drive mounted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Put designed structures in:  My Drive/{drive_folder}/inputs")
print(f"Results will be saved to:    My Drive/{drive_folder}/outputs")
print(f"Weights are cached in:       My Drive/{drive_folder}/tool_weights\n")

found = sorted(d for d in os.listdir(DRIVE_INPUTS)
               if os.path.isdir(os.path.join(DRIVE_INPUTS, d)))
loose = sorted(f for f in os.listdir(DRIVE_INPUTS)
               if f.lower().endswith((".pdb", ".cif")))
if found:
    print("Subfolders in your inputs folder - paste one of these names into")
    print("`drive_subfolder` in step 6:")
    for d in found:
        n = len([f for f in os.listdir(os.path.join(DRIVE_INPUTS, d))
                 if f.lower().endswith((".pdb", ".cif"))])
        print(f"    {d}   ({n} structures)")
if loose:
    print(f"\n{len(loose)} structure(s) sit directly in inputs/ - leave")
    print("`drive_subfolder` empty in step 6 to use those.")
if not found and not loose:
    print("inputs folder is empty for now. A whole evaluation runs on a *folder* of")
    print("designs, so drop your .pdb or .cif files in there (all against the same")
    print("target, same chain naming) and re-run this cell to list them.")

In [ ]:
#@title 3. Set up everything (~25 min, once per runtime)
#@markdown Everything the later steps need, in one cell: Python 3.11, the pinned
#@markdown PyTorch / JAX / NumPy stack, both upstream repos, and every weight and
#@markdown checkpoint — so a design or evaluation run is pure compute and never stops
#@markdown halfway to fetch something.
#@markdown
#@markdown Safe to re-run: each phase checks whether it is already done, and partial
#@markdown downloads resume. It verifies itself at the end and **raises** rather than
#@markdown let a broken environment reach the steps below.
install_for = "binder: design and evaluate" #@param ["binder: design and evaluate", "binder: evaluate only", "monomer", "everything"]
protenix_filter = "mini (0.5 GB)" #@param ["none - AlphaFold2 only", "mini (0.5 GB)", "full + template (2 GB)", "both (2.5 GB)"]
af2_params = "model 1 only (~1.1 GB)" #@param ["model 1 only (~1.1 GB)", "all five models (~5.3 GB)"]
cache_in_drive = False #@param {type:"boolean"}
#@markdown <small>`install_for` decides which folding models are fetched: the binder
#@markdown pipeline refolds with AlphaFold2, the monomer pipeline with ESMFold.
#@markdown
#@markdown `protenix_filter` is the second opinion on the binder interface, and the
#@markdown largest download here. `mini` is what step 6 uses by default; `full` is
#@markdown what step 5's `extended` mode uses and brings the template model with it.
#@markdown Pick `none` to start with AlphaFold2 only, which is upstream's `preview`
#@markdown mode, and come back for the rest later.
#@markdown
#@markdown `model 1 only` is all the default settings use — upstream's
#@markdown `af2.model_ids` is `[0]`. Either way the full 5.3 GB archive is downloaded
#@markdown and read; the choice is how much of it is kept.
#@markdown
#@markdown **By default nothing is written to your Drive.** Tick `cache_in_drive` to
#@markdown keep the weights in `My Drive/PXDesign/tool_weights`: it then looks there
#@markdown first, and only downloads — and saves back — what is missing. Worth it from
#@markdown the second session on, since this is ~5 GB of downloads.</small>
pxdbench_ref = "main" #@param {type:"string"}
pxdesign_ref = "main" #@param {type:"string"}
#@markdown <small>The branch, tag or commit of each repo. Leave both at `main` to
#@markdown track upstream; pin commits if you need a run reproducible months from
#@markdown now.</small>

import os, shutil, subprocess, textwrap, time

ENV  = "/content/pxdenv"
PY   = f"{ENV}/bin/python"
PIP  = f"{ENV}/bin/pip"
REPO = "/content/PXDesignBench"
GEN  = "/content/PXDesign"
WEIGHTS = "/content/tool_weights"
AF2_DIR, MPNN_DIR, ESM_DIR = f"{WEIGHTS}/af2", f"{WEIGHTS}/mpnn", f"{WEIGHTS}/esmfold"
CCD_DIR, CKPT_DIR = f"{WEIGHTS}/ccd_cache", f"{WEIGHTS}/checkpoint"
ENV_STAMP = "/content/.pxd_env_ok"

need_gen  = install_for in ("binder: design and evaluate", "everything")
need_af2  = install_for.startswith("binder") or install_for == "everything"
need_esm  = install_for in ("monomer", "everything")
want_mini = (protenix_filter.startswith(("mini", "both"))
             or install_for == "everything")
want_full = (protenix_filter.startswith(("full", "both"))
             or install_for == "everything")

_t0 = time.time()
_n = 0
def phase(what):
    global _n
    _n += 1
    print(f"\n{'=' * 70}")
    print(f"[{_n}/5]  {what}   ({(time.time() - _t0) / 60:.0f} min in)")
    print("=" * 70)

# ---- Drive cache ----------------------------------------------------------
DRIVE_WEIGHTS = None
if cache_in_drive and os.path.isfile("/content/.pxd_drive"):
    DRIVE_WEIGHTS = open("/content/.pxd_drive").read().strip() + "/tool_weights"
    os.makedirs(DRIVE_WEIGHTS, exist_ok=True)
    with open("/content/.pxd_weights_cache", "w") as f:
        f.write(DRIVE_WEIGHTS)
elif cache_in_drive:
    print("Drive is not mounted (step 2) - downloading to local disk only.")
if not DRIVE_WEIGHTS and os.path.exists("/content/.pxd_weights_cache"):
    os.remove("/content/.pxd_weights_cache")

def restore(sub):
    # Copy a weights subfolder back from Drive, if a previous session left one.
    if not DRIVE_WEIGHTS:
        return
    src = os.path.join(DRIVE_WEIGHTS, sub)
    if os.path.isdir(src) and os.listdir(src):
        print(f"  restoring {sub} from Drive...")
        subprocess.run(["rsync", "-a", src + "/", os.path.join(WEIGHTS, sub) + "/"])

def stash(sub):
    if not DRIVE_WEIGHTS:
        return
    print(f"  copying {sub} to Drive...")
    subprocess.run(["rsync", "-a", os.path.join(WEIGHTS, sub) + "/",
                    os.path.join(DRIVE_WEIGHTS, sub) + "/"])

def fetch(url, path, label):
    # Sized against the server rather than a hardcoded number, so a download that
    # was cut off half way is resumed instead of being mistaken for a complete
    # file - which would hand a truncated checkpoint to torch.load later.
    want = 0
    try:
        _h = subprocess.run(["curl", "-sIL", "--max-time", "30", url],
                            capture_output=True, text=True).stdout
        for _l in _h.splitlines():
            if _l.lower().startswith("content-length:"):
                want = int(_l.split(":", 1)[1].strip())
    except Exception:
        pass
    have = os.path.getsize(path) if os.path.isfile(path) else 0
    if want and have == want:
        print(f"  have    {label}  ({have / 1e6:.0f} MB)")
        return
    if have > want > 0:
        os.remove(path)          # longer than the server's copy: start over
        have = 0
    if have:
        print(f"  resume  {label}  ({have / 1e6:.0f} of {want / 1e6:.0f} MB)")
    else:
        print(f"  get     {label}" + (f"  ({want / 1e6:.0f} MB)" if want else ""))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    !curl -L -C - --retry 3 --progress-bar -o {path} {url}

# ========================================================= 1. Python 3.11
phase("Python 3.11, alongside Colab's own")
if shutil.which("python3.11") and shutil.which("rsync"):
    print("already installed.")
else:
    # PXDesignBench pins PyTorch 2.3.1 and JAX 0.4.29, neither of which has wheels
    # for the Python Colab ships. rsync is for the Drive copying.
    !apt-get update -qq
    !apt-get install -y -qq python3.11 python3.11-venv python3.11-dev rsync
print(subprocess.run(["python3.11", "--version"],
                     capture_output=True, text=True).stdout.strip())

# ========================================================= 2. the stack
phase("The pinned stack and both upstream repos (~12 min, ~12 GB)")

# A half-built environment is worse than none: rebuild rather than install on top.
if not os.path.isfile(PY):
    print("creating environment...")
    os.system(f"rm -rf {ENV}")
    os.system(f"python3.11 -m venv {ENV}")
assert os.path.isfile(PY), "venv creation failed - see the apt output above"

# Colab exports PYTHONPATH=/env/python, which would leak Colab's own packages
# (NumPy 2, a different torch) into this environment. Every subprocess below is
# launched with it replaced by the repo itself.
PXD_ENV = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
PXD_ENV["PATH"] = f"{ENV}/bin:" + os.environ.get("PATH", "")
PXD_ENV["PYTHONPATH"] = REPO
PXD_ENV["PYTHONNOUSERSITE"] = "1"
PXD_ENV["VIRTUAL_ENV"] = ENV

if os.path.isfile(ENV_STAMP):
    print(f"already built - delete {ENV_STAMP} to force a rebuild.")
else:
    !{PIP} install -q --upgrade pip setuptools wheel
    !{PIP} install -q ninja

    # 1) GPU PyTorch first: deepspeed, pulled in by Protenix below, imports torch
    #    while it installs itself and fails outright if torch is not already there.
    !{PIP} install -q "torch==2.3.1" --index-url https://download.pytorch.org/whl/cu121

    # 2) Protenix - the structure predictor both repos build on.
    !DS_BUILD_OPS=0 {PIP} install -q "git+https://github.com/bytedance/Protenix.git@v0.5.0+pxd"

    # 3) PXDesignBench's own dependencies.
    !{PIP} install -q einops natsort dm-tree posix_ipc "transformers==4.51.3" "dm-haiku==0.0.13" "optax==0.2.5"

    # 4) ColabDesign, which is how AlphaFold2 is run here. --no-deps on purpose:
    #    its dependency list would drag in a different JAX than the one pinned next.
    !{PIP} install -q git+https://github.com/sokrypton/ColabDesign.git --no-deps

    # 5) JAX with CUDA. The wheel on jax_cuda_releases bundles its own CUDA and
    #    cuDNN, so it does not fight PyTorch over the nvidia-* pip packages. If that
    #    index is ever pulled, fall back to the plugin build from PyPI.
    !{PIP} install -q "jax[cuda]==0.4.29" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    _gpu_jax = subprocess.run(
        [PY, "-c", "import jax, sys; "
                   "sys.exit(0 if any(d.platform == 'gpu' for d in jax.devices()) else 1)"],
        env=PXD_ENV, capture_output=True)
    if _gpu_jax.returncode != 0:
        print("\nCUDA jaxlib not installed from jax_cuda_releases - falling back")
        !{PIP} install -q "jax[cuda12]==0.4.29"

    # 6) Compatibility pins. pip resolves these to their newest releases, which now
    #    expect jax 0.7 and NumPy 2 and would fail at import under the pins above:
    #    chex arrives through optax, and ColabDesign reaches it through optax too.
    #    immutabledict is declared by ColabDesign but never imported - installed
    #    anyway so its warning does not mask a real one.
    !{PIP} install -q "chex==0.1.87" "ml_dtypes==0.4.0" immutabledict

    # 7) Pinned last so nothing drags NumPy 2 back in: this stack is built on 1.x.
    !{PIP} install -q "numpy==1.26.3"

    # 8) PXDesignBench - the evaluator.
    if not os.path.isdir(REPO):
        !git clone -q https://github.com/bytedance/PXDesignBench.git {REPO}
    !cd {REPO} && git fetch -q origin "+refs/heads/*:refs/remotes/origin/*" --tags 2>/dev/null; cd {REPO} && git checkout -q {pxdbench_ref} 2>/dev/null || git checkout -q -B {pxdbench_ref} origin/{pxdbench_ref} 2>/dev/null; cd {REPO} && git pull -q --ff-only 2>/dev/null; true
    !{PIP} install -q -e {REPO}
    !chmod +x {REPO}/pxdbench/metrics/TMalign

    # 9) PXDesign - the generator. Everything above is exactly what PXDesign's own
    #    install.sh installs before this line, so it drops on top without moving a
    #    pin: its setup.py declares no dependencies at all, and the only package it
    #    imports that nothing else pulls in is click.
    if not os.path.isdir(GEN):
        !git clone -q https://github.com/bytedance/PXDesign.git {GEN}
    !cd {GEN} && git fetch -q origin "+refs/heads/*:refs/remotes/origin/*" --tags 2>/dev/null; cd {GEN} && git checkout -q {pxdesign_ref} 2>/dev/null || git checkout -q -B {pxdesign_ref} origin/{pxdesign_ref} 2>/dev/null; cd {GEN} && git pull -q --ff-only 2>/dev/null; true
    !{PIP} install -q click
    !{PIP} install -q -e {GEN}

print("\n--- environment check ---")
_check = textwrap.dedent('''
    import sys, torch, numpy, jax, jaxlib, colabdesign, protenix, pxdbench, transformers
    import chex, optax
    # What the run steps actually import: AlphaFold2 and ProteinMPNN both come
    # through these, and they reach jax via optax/chex rather than directly.
    from colabdesign import clear_mem, mk_afdesign_model
    from colabdesign.mpnn import mk_mpnn_model
    print('python  :', sys.version.split()[0])
    print('torch   :', torch.__version__)
    print('numpy   :', numpy.__version__)
    print('jax     :', jax.__version__)
    print('jaxlib  :', jaxlib.__version__)
    print('cuda    :', torch.cuda.is_available())
    print('gpu     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
    print('jaxdev  :', jax.devices()[0].platform)
    print('torch<->numpy:', torch.tensor([1.0]).numpy().sum())
    from pxdbench.tasks import get_task_class
    print('chex    :', chex.__version__)
    print('af2+mpnn: importable')
    print('tasks   :', get_task_class('binder').__name__, get_task_class('monomer').__name__)
    import pxdesign
    from pxdesign.utils import inputs as _pxd_inputs
    print('pxdesign: importable')
''')
with open("/content/_check.py", "w") as f:
    f.write(_check)
_chk = subprocess.run([PY, "-u", "/content/_check.py"], env=PXD_ENV,
                      capture_output=True, text=True)
print(_chk.stdout + _chk.stderr)

# Verified rather than left for the eye, so an unattended Run all halts on the real
# cause instead of carrying a broken environment into the steps below.
_vals = {k.strip(): v.strip()
         for k, v in (l.split(":", 1) for l in _chk.stdout.splitlines() if ":" in l)}
_bad = []
if _chk.returncode:
    _bad.append("the check script itself failed - see its output above")
if not _vals.get("numpy", "").startswith("1."):
    _bad.append(f"numpy is {_vals.get('numpy', '?')}, must be 1.26.3 - something "
                "reinstalled NumPy 2 over the pin")
if not _vals.get("torch", "").startswith("2.3.1"):
    _bad.append(f"torch is {_vals.get('torch', '?')}, must be 2.3.1")
if _vals.get("cuda") != "True":
    _bad.append("torch cannot see the GPU - check the runtime type (step 1)")
if _vals.get("af2+mpnn") != "importable":
    _bad.append("ColabDesign will not import, so AlphaFold2 and ProteinMPNN cannot "
                "run - usually chex or ml_dtypes resolved to a release that wants "
                "jax 0.7 / NumPy 2")
if _vals.get("jaxdev") != "gpu":
    _bad.append("JAX is on CPU, so AlphaFold2 would take hours per design - the "
                "CUDA jaxlib did not install")
if _vals.get("pxdesign") != "importable":
    _bad.append("PXDesign (the generator) will not import, so step 5 cannot run")

if _bad:
    if os.path.exists(ENV_STAMP):
        os.remove(ENV_STAMP)
    print("\n*** ENVIRONMENT IS NOT USABLE ***")
    for _b in _bad:
        print("  - " + _b)
    raise RuntimeError(
        f"Environment check failed ({len(_bad)} problem(s), listed above). "
        "Do NOT re-run this cell on top of the failure: Runtime > Disconnect and "
        "delete runtime, then start again from step 1.")
open(ENV_STAMP, "w").close()
_head = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"],
                       capture_output=True, text=True).stdout.strip()
_ghead = subprocess.run(["git", "-C", GEN, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"PXDesignBench at {_head} ({pxdbench_ref}), PXDesign at {_ghead} ({pxdesign_ref}).")

# ========================================================= 3. fused LayerNorm
phase("Fused LayerNorm kernel (optional; a speed-up, nothing more)")
# Protenix ships a fused LayerNorm CUDA kernel and uses it when
# LAYERNORM_TYPE=fast_layernorm is set, which is what upstream's demos set. It is
# the same arithmetic as torch.nn.LayerNorm in one kernel. On current Colab it
# usually does NOT build: the CUDA toolkit has dropped the compute_70 target
# Protenix hardcodes. That costs some speed and nothing else.
_ln = subprocess.run(
    [PY, "-c", "from protenix.model.layer_norm.layer_norm import FusedLayerNorm; "
               "print('fast_layernorm OK')"],
    env=PXD_ENV, capture_output=True, text=True)
FASTLN_OK = "fast_layernorm OK" in _ln.stdout
flag = "/content/.pxd_fastln_ok"
if FASTLN_OK:
    open(flag, "w").close()
    print("built - the run steps will use it, as upstream does.")
else:
    if os.path.exists(flag):
        os.remove(flag)
    _why = [l for l in _ln.stderr.splitlines() if "fatal" in l.lower()]
    print("not built" + (f": {_why[0].strip()}" if _why else ""))
    print("Falling back to PyTorch's own LayerNorm - same numbers, somewhat slower.")

# ========================================================= 4. weights
phase("Model weights")
for _d in (AF2_DIR, MPNN_DIR, ESM_DIR, CCD_DIR, CKPT_DIR):
    os.makedirs(_d, exist_ok=True)
if DRIVE_WEIGHTS:
    print(f"Drive cache on: "
          f"{DRIVE_WEIGHTS.replace('/content/drive/MyDrive', 'My Drive')}")
else:
    print("Drive cache off: these go to Colab's disk and leave with the runtime.")

# -- the chemical component dictionary: needed by everything that reads a structure,
#    and step 4 reads it without being able to download it for itself.
restore("ccd_cache")
_ccd_base = "https://pxdesign.tos-cn-beijing.volces.com/release_data"
_ccd = ["components.v20240608.cif",
        "components.v20240608.cif.rdkit_mol.pkl",
        "clusters-by-entity-40.txt"]
_before = sorted(os.listdir(CCD_DIR))
for _f in _ccd:
    fetch(f"{_ccd_base}/{_f}", f"{CCD_DIR}/{_f}", f"ccd/{_f}")
if sorted(os.listdir(CCD_DIR)) != _before:
    stash("ccd_cache")

# -- AlphaFold2 parameters
if need_af2:
    restore("af2")
    want = ["params_model_1.npz", "params_model_1_ptm.npz"]
    if af2_params.startswith("all five"):
        want = [f"params_model_{i}{s}.npz" for i in range(1, 6)
                for s in ("", "_ptm", "_multimer_v3")]
    missing = [w for w in want if not os.path.isfile(f"{AF2_DIR}/{w}")]
    if missing:
        url = "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar"
        # Piped straight into tar, selecting members: the archive is 5.3 GB and
        # there is no reason to land it on disk before throwing most of it away.
        members = ("--wildcards 'params_model_1*.npz' LICENSE"
                   if not af2_params.startswith("all five") else "")
        print(f"  get     af2 parameters ({len(missing)} file(s) missing, 5.3 GB read)")
        !curl -sL {url} | tar -x -C {AF2_DIR} {members} || true
        stash("af2")
    else:
        print("  have    af2 parameters")

# -- ProteinMPNN
restore("mpnn")
if not os.path.isdir(f"{MPNN_DIR}/vanilla_model_weights"):
    print("  get     ProteinMPNN weights (~50 MB)")
    !rm -rf /content/_mpnn && git clone -q --depth 1 https://github.com/dauparas/ProteinMPNN.git /content/_mpnn
    for sub in ("ca_model_weights", "soluble_model_weights", "vanilla_model_weights"):
        if not os.path.isdir(f"{MPNN_DIR}/{sub}"):
            shutil.copytree(f"/content/_mpnn/{sub}", f"{MPNN_DIR}/{sub}")
    shutil.rmtree("/content/_mpnn", ignore_errors=True)
    stash("mpnn")
else:
    print("  have    ProteinMPNN weights")

# -- ESMFold
if need_esm:
    restore("esmfold")
    if not os.path.isfile(f"{ESM_DIR}/pytorch_model.bin"):
        print("  get     ESMFold (~2.7 GB)")
        !{PY} -c "from huggingface_hub import snapshot_download; snapshot_download('facebook/esmfold_v1', local_dir='{ESM_DIR}', allow_patterns=['*.json','*.txt','*.bin','*.model'])"
        stash("esmfold")
    else:
        print("  have    ESMFold")

# ========================================================= 5. checkpoints
phase("Model checkpoints")
# Fetched here rather than left to the runs, which is what made a first design or
# evaluation stop for minutes in the middle. One directory serves all three
# consumers: pxdesign takes --load_checkpoint_dir, and PXDesignBench's two Protenix
# tools each take their own load_checkpoint_dir, which steps 5 and 6 pass.
restore("checkpoint")
_ckpt_base = "https://pxdesign.tos-cn-beijing.volces.com/release_model"
_ckpts = []
if need_gen:
    _ckpts.append("pxdesign_v0.1.0")
if want_mini:
    _ckpts.append("protenix_mini_default_v0.5.0")
if want_full:
    _ckpts.append("protenix_base_default_v0.5.0")
    # Switched to automatically by `extended` mode when the target folds close
    # enough to its own structure to be used as a template.
    _ckpts.append("protenix_mini_tmpl_v0.5.0")
if not _ckpts:
    print("none needed for this selection.")
_before = sorted(os.listdir(CKPT_DIR))
for _c in _ckpts:
    fetch(f"{_ckpt_base}/{_c}.pt", f"{CKPT_DIR}/{_c}.pt", _c)
if sorted(os.listdir(CKPT_DIR)) != _before:
    stash("checkpoint")

# ========================================================= verify
print(f"\n{'=' * 70}")
print("weights check")
print("=" * 70)
required = [f"{CCD_DIR}/{f}" for f in _ccd]
if need_af2:
    required += [f"{AF2_DIR}/params_model_1.npz", f"{AF2_DIR}/params_model_1_ptm.npz"]
if need_esm:
    required += [f"{ESM_DIR}/pytorch_model.bin", f"{ESM_DIR}/config.json"]
required += [f"{MPNN_DIR}/vanilla_model_weights/v_48_020.pt",
             f"{MPNN_DIR}/ca_model_weights/v_48_020.pt"]
required += [f"{CKPT_DIR}/{c}.pt" for c in _ckpts]

missing = [p for p in required if not os.path.isfile(p)]
for p in required:
    if os.path.isfile(p):
        print(f"  ok    {os.path.relpath(p, WEIGHTS)}  "
              f"({os.path.getsize(p) / 1e6:.0f} MB)")
    else:
        print(f"  MISS  {os.path.relpath(p, WEIGHTS)}")
if missing:
    raise RuntimeError(
        f"{len(missing)} file(s) did not download. Most often a full disk - check "
        "the resources pane - or a dropped connection. Re-run this cell: what is "
        "already there is kept, and a partial download resumes.")

_total = sum(os.path.getsize(os.path.join(r, f))
             for r, _, fs in os.walk(WEIGHTS) for f in fs)
print(f"\nSetup done in {(time.time() - _t0) / 60:.0f} min. "
      f"{_total / 1e9:.1f} GB of weights in {WEIGHTS}.")
print("Nothing below downloads a model - go to step 4 to design, or step 6 to")
print("evaluate designs you already have.")

**Check step 3's output before continuing.** The environment check wants
`torch: 2.3.1+cu121`, `numpy: 1.26.3`, `cuda: True`, a GPU name, `jaxdev: gpu`, and
both `af2+mpnn` and `pxdesign` reading `importable`. The weights check should end in
a size and no `MISS` lines.

`jaxdev: cpu` is the one to watch for. It does not raise on its own in every case and
AlphaFold2 on CPU looks like a hang rather than an error — hours per design instead of
about a minute.

In [ ]:
#@title 4. Describe the design target (crop, hotspots)
#@markdown Writes the input YAML `pxdesign` reads, then runs upstream's own two checks
#@markdown on it: `check-input` for the format, and `parse-target`, which writes out
#@markdown exactly the structure the model will see, so `crop` and `hotspots` can be
#@markdown verified instead of assumed.
#@markdown
#@markdown Needed only for step 5. Skip 4 and 5 entirely if you came to evaluate
#@markdown designs you already have.
task_name = "design1" #@param {type:"string"}
target_from = "bundled example (PD-L1)" #@param ["bundled example (PD-L1)", "Google Drive file", "path on disk"]
target_path = "" #@param {type:"string"}
binder_length = 80 #@param {type:"integer"}
#@markdown <small>`binder_length` is the number of residues to design. PXDesign's own
#@markdown advice is to run several lengths rather than guess one.</small>

#@markdown ---
#@markdown ### Target chain
target_chain = "A" #@param {type:"string"}
crop = "1-116" #@param {type:"string"}
hotspots = "40,99,107" #@param {type:"string"}
msa_dir = "" #@param {type:"string"}
other_chains = "" #@param {type:"string"}
show_structure = True #@param {type:"boolean"}
#@markdown <small>**Residue numbers are mmCIF `label_seq_id`** — 1-based and strictly
#@markdown sequential, the number Mol\* calls *Sequence ID*, **not** *Auth ID*. The
#@markdown structure this cell prints is the way to confirm you picked the residues you
#@markdown meant.
#@markdown
#@markdown `crop` keeps only part of the chain: `1-116`, or `1-50,80-100` for two
#@markdown pieces; empty keeps all of it. `hotspots` is where the binder is aimed and
#@markdown is optional, but without it diffusion is free to land anywhere on the
#@markdown target. `msa_dir` must hold `pairing.a3m` and `non_pairing.a3m`; left empty,
#@markdown step 5 searches an MSA online, which is slower and rate-limited. Name any
#@markdown further target chains in `other_chains` (`B,C`) to keep them whole — for a
#@markdown second chain that needs its own crop or hotspots, edit the YAML this cell
#@markdown writes and re-run step 5.</small>

import os, re, shutil, subprocess

ENV, GEN = "/content/pxdenv", "/content/PXDesign"
PXD, WEIGHTS = f"{ENV}/bin/pxdesign", "/content/tool_weights"
assert os.path.isfile(PXD), "run step 3 first"
# No model weights needed to write and check a YAML, but parsing the target reads
# the chemical component dictionary, and parse-target does not download it the way a
# run would - so step 3 has to have fetched it.
assert os.path.isfile(f"{WEIGHTS}/ccd_cache/components.v20240608.cif"), (
    "the chemical component dictionary is missing - run step 3 first; parsing a "
    "target needs it and this step cannot download it")

safe_task = re.sub(r"[^A-Za-z0-9._-]", "_", task_name) or "design1"
work = f"/content/pxd_design/{safe_task}"
os.makedirs(work, exist_ok=True)

# ---- the target structure ------------------------------------------------
if target_from.startswith("bundled"):
    src = f"{GEN}/examples/5o45.cif"
    if not msa_dir.strip():
        msa_dir = f"{GEN}/examples/msa/PDL1/0"
elif target_from == "Google Drive file":
    assert os.path.isfile("/content/.pxd_drive"), "Drive is not mounted - run step 2"
    src = os.path.join(open("/content/.pxd_drive").read().strip(),
                       "inputs", target_path.strip())
    assert os.path.isfile(src), f"no such file under My Drive/PXDesign/inputs: {target_path!r}"
else:
    src = target_path.strip()
    assert os.path.isfile(src), f"not a file: {src!r}"

ext = os.path.splitext(src)[1].lower()
assert ext in (".cif", ".pdb"), f"the target must be .cif or .pdb, not {ext!r}"
if ext == ".pdb":
    print("note: .pdb input is converted to mmCIF by pxdesign, and chain and residue")
    print("      ids may be reassigned in the process. Upstream recommends .cif.\n")
# Copied under /content with a space-free name: pxdesign passes its arguments to
# Protenix's config parser, which re-splits the command line on whitespace.
target = f"{work}/target{ext}"
shutil.copy2(src, target)

# ---- the YAML ------------------------------------------------------------
_list = lambda s: [x.strip() for x in s.split(",") if x.strip()]

chain = target_chain.strip()
assert chain, "name the target chain"
hot = [int(x) for x in _list(hotspots)]

y = ["target:", f"  file: {target}", "  chains:", f"    {chain}:"]
if crop.strip():
    y.append("      crop: [" + ", ".join(f'"{c}"' for c in _list(crop)) + "]")
if hot:
    y.append("      hotspots: [" + ", ".join(str(h) for h in hot) + "]")
if msa_dir.strip():
    assert os.path.isdir(msa_dir.strip()), f"no such MSA folder: {msa_dir.strip()!r}"
    y.append(f"      msa: {msa_dir.strip()}")
for c in _list(other_chains):
    y.append(f'    {c}: "all"')
y += ["", f"binder_length: {int(binder_length)}", f"task_name: {safe_task}", ""]

yaml_path = f"{work}/{safe_task}.yaml"
with open(yaml_path, "w") as f:
    f.write("\n".join(y))
print(f"--- {yaml_path} ---")
print("\n".join(y))

# ---- upstream's own validation -------------------------------------------
env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
env.update({
    "PATH": f"{ENV}/bin:" + os.environ.get("PATH", ""),
    "PYTHONPATH": GEN,
    "PYTHONNOUSERSITE": "1",
    "VIRTUAL_ENV": ENV,
    "TOOL_WEIGHTS_ROOT": WEIGHTS,
    # Where step 3 put the chemical component dictionary.
    "PROTENIX_DATA_ROOT_DIR": f"{WEIGHTS}/ccd_cache",
})

for _step, _args, _why in [
    ("check-input", ["--yaml", yaml_path],
     "the YAML above is rejected - a field is missing or malformed"),
    ("parse-target", ["--yaml", yaml_path, "-o", f"{work}/parsed"],
     "the target could not be parsed - usually the chain id does not exist in this "
     "file, or the crop range runs past the end of the chain"),
]:
    print(f"\n--- {_step} ---")
    _r = subprocess.run([PXD, _step] + _args, env=env, cwd=GEN,
                        capture_output=True, text=True)
    print((_r.stdout + _r.stderr).strip())
    if _r.returncode != 0:
        raise RuntimeError(_why)

parsed = f"{work}/parsed/{safe_task}_parsed_target.cif"
assert os.path.isfile(parsed), f"parse-target wrote no structure to {parsed}"

# ---- what the hotspot numbers actually point at --------------------------
# Read back from the parsed file rather than the input, because the parsed file is
# the structure the model sees: a hotspot that was cropped away is simply absent.
if hot:
    hdr, found = [], {}
    with open(parsed) as f:
        for line in f:
            if line.startswith("_atom_site."):
                hdr.append(line.strip().split(".")[1])
            elif hdr and line.startswith(("ATOM", "HETATM")):
                p = line.split()
                if len(p) < len(hdr):
                    continue
                r = dict(zip(hdr, p))
                # label_seq_id only: that is the numbering PXDesign documents using,
                # and reading auth_seq_id too would happily report a residue the
                # model never sees under that number.
                try:
                    n = int(r.get("label_seq_id", "."))
                except ValueError:
                    continue
                if n in hot:
                    found.setdefault(n, f"{r['label_comp_id']} "
                                        f"(chain {r['label_asym_id']})")
    print("\n--- hotspots in the parsed target ---")
    _missing = [h for h in hot if h not in found]
    for h in hot:
        print(f"  {str(h).rjust(5)} : {found.get(h, 'NOT FOUND')}")
    if _missing:
        print(f"\n  {len(_missing)} hotspot(s) are not in the structure the model will")
        print("  see. Either the number is past the end of the chain, or the crop")
        print("  removed it, or it is an auth_seq_id rather than a label_seq_id.")
        print("  Fix this before step 5 - diffusion cannot aim at a residue that")
        print("  is not there.")

if show_structure:
    try:
        import py3Dmol
    except ImportError:
        !pip install -q py3Dmol
        import py3Dmol
    # The same colouring as the .pml upstream writes next to the parsed file:
    # kept region marine, hotspots pink sticks, everything else grey.
    v = py3Dmol.view(width=900, height=600)
    v.addModel(open(parsed).read(), "cif")
    v.setStyle({"cartoon": {"color": "grey70"}})
    if crop.strip():
        for rng in _list(crop):
            v.setStyle({"chain": chain, "resi": rng},
                       {"cartoon": {"color": "marine"}})
    if hot:
        sel = {"chain": chain, "resi": [str(h) for h in hot]}
        v.addStyle(sel, {"stick": {"colorscheme": "pinkCarbon", "radius": 0.3}})
        v.addStyle(sel, {"cartoon": {"color": "pink"}})
    v.zoomTo()
    v.show()
    print("marine = kept by crop, pink sticks = hotspots, grey = the rest")

print(f"\nTarget ready. Step 5 reads {yaml_path}")
print(f"PyMOL script, if you want a closer look: {work}/parsed/{safe_task}_parsed_target.pml")

In [ ]:
#@title 5. Generate binders with PXDesign-d
#@markdown Runs the diffusion generator on the target from step 4 — this is the half
#@markdown that invents backbones. Calls upstream's `pxdesign` CLI with the arguments
#@markdown its own README gives; nothing here is reimplemented.
task_name = "design1" #@param {type:"string"}
mode = "pipeline: preview (AF2 filter)" #@param ["infer: backbones only", "pipeline: preview (AF2 filter)", "pipeline: extended (AF2 + Protenix)"]
n_sample = 10 #@param {type:"integer"}
seed = 2025 #@param {type:"integer"}
#@markdown <small>`infer` generates backbones and stops — cheapest, and step 6 can
#@markdown score them afterwards with every setting you can see. `preview` adds
#@markdown upstream's AF2-IG filter and ranking. `extended` adds the full Protenix
#@markdown model on top; it is what the paper used, and the heaviest thing in this
#@markdown notebook by a wide margin.
#@markdown
#@markdown `n_sample` is how many backbones to draw. Upstream's own presets use 100
#@markdown (preview) and 500 (extended) and recommend 10000+ for a real campaign; 10 is
#@markdown set here so a first run finishes. A T4 is not the machine for thousands —
#@markdown raise this as far as your session allows and run it more than once with
#@markdown different seeds.</small>

#@markdown ---
#@markdown ### Performance
dtype = "fp32" #@param ["fp32", "bf16"]
n_step = 400 #@param {type:"integer"}
n_max_runs = 1 #@param {type:"integer"}
quiet_output = True #@param {type:"boolean"}
#@markdown <small>`bf16` needs an Ampere card or newer (A100, L4) — on a T4 it is not
#@markdown supported and `fp32` is the only option. `n_step` is the number of diffusion
#@markdown steps; 400 is upstream's default and lowering it trades quality for speed.
#@markdown DeepSpeed's evoformer attention stays off: it needs a CUTLASS build this
#@markdown environment does not have.</small>

import codecs, os, re, subprocess, sys, time

ENV, GEN = "/content/pxdenv", "/content/PXDesign"
PXD, WEIGHTS = f"{ENV}/bin/pxdesign", "/content/tool_weights"
assert os.path.isfile(PXD), "run step 4 first"
assert os.path.isfile(f"{WEIGHTS}/af2/params_model_1.npz"), "run step 6 first"

safe_task = re.sub(r"[^A-Za-z0-9._-]", "_", task_name) or "design1"
work = f"/content/pxd_design/{safe_task}"
yaml_path = f"{work}/{safe_task}.yaml"
assert os.path.isfile(yaml_path), (
    f"no design target at {yaml_path} - run step 4 first (with the same task_name)")
out_dir = f"{work}/out"
os.makedirs(out_dir, exist_ok=True)

# Upstream defaults use_fast_ln to True. Step 3 is where the kernel it wants is
# either built or not; if it was not, say so rather than let the generator fail
# on a missing kernel.
fast_ln = os.path.exists("/content/.pxd_fastln_ok")

sub = "infer" if mode.startswith("infer") else "pipeline"
cmd = [PXD, sub,
       "-i", yaml_path,
       "-o", out_dir,
       "--N_sample", str(int(n_sample)),
       "--N_step", str(int(n_step)),
       "--dtype", dtype,
       "--seeds", str(int(seed)),
       "--use_fast_ln", "true" if fast_ln else "false",
       "--load_checkpoint_dir", f"{WEIGHTS}/checkpoint",
       # Upstream defaults to 16 dataloader workers, which suits the machines it was
       # written for. Colab gives you two cores, and torch warns that oversubscribing
       # them can slow the loader down or wedge it.
       "--num_workers", str(max(1, min(4, os.cpu_count() or 2)))]
if sub == "pipeline":
    cmd += ["--preset", "extended" if "extended" in mode else "preview",
            "--N_max_runs", str(int(n_max_runs)),
            "--use_deepspeed_evo_attention", "false"]
    # --seeds must hold exactly N_max_runs values, so derive the rest from the one
    # the form gives. Upstream asserts this rather than padding.
    if int(n_max_runs) > 1:
        cmd[cmd.index("--seeds") + 1] = ",".join(
            str(int(seed) + i) for i in range(int(n_max_runs)))

env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
env.update({
    "PATH": f"{ENV}/bin:" + os.environ.get("PATH", ""),
    "PYTHONPATH": GEN,
    "PYTHONNOUSERSITE": "1",
    "VIRTUAL_ENV": ENV,
    # check_tool_weights() reads this, and so do the AF2 and ProteinMPNN filters it
    # runs through PXDesignBench.
    "TOOL_WEIGHTS_ROOT": WEIGHTS,
    "PROTENIX_DATA_ROOT_DIR": f"{WEIGHTS}/ccd_cache",
    "USE_DEEPSPEED_EVO_ATTENTION": "false",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
    "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.5",
})
if fast_ln:
    env["LAYERNORM_TYPE"] = "fast_layernorm"

print("command:")
print("  " + " \\\n    ".join(cmd[1:]))
print(f"\ntarget : {yaml_path}")
print(f"output : {out_dir}")
print(f"fused LayerNorm: {'on' if fast_ln else 'off (torch LayerNorm)'}")
print(f"\n{int(n_sample)} backbone(s) to generate" +
      ("" if sub == "infer" else ", then filtered") + ".")
print("The first run also downloads the PXDesign diffusion checkpoint and the")
print("Protenix checkpoints it filters with (~1 GB and up).")
print("-" * 70)

_NOISE = re.compile(
    r"PROTENIX_DATA_ROOT_DIR not set"
    r"|xla_bridge"
    r"|Featurizing "
    r"|Loaded constraint feature"
    r"|use_msa is \w+ for sequence"
    r"|No need to perform chain permutation"
    r"|Rebuilt and renumbered PDB saved"
    r"|WARNING: 'model_\d_ptm' not found"
    r"|Run subprocess success: 0"
    r"|Distributed environment:|LOCAL_RANK:|Finished init ENV"
    r"|train scheduler |Sampled key:|Loading protenix filter model"
    r"|^\{[\d:, ]+\} [\d.]+$"
)
_STAMP = re.compile(r"^\d{4}-\d\d-\d\d \d\d:\d\d:\d\d,\d+ \[[^\]]+\] \w+ [\w.]+: ")
_STAMP2 = re.compile(r"^\d{4}-\d\d-\d\d \d\d:\d\d:\d\d,\d+ \w+\s+\[[^\]]+\] ")

def keep(line):
    if not quiet_output:
        return line
    body = _STAMP2.sub("", _STAMP.sub("", line)).rstrip()
    if not body.strip() or _NOISE.search(line) or _NOISE.search(body):
        return None
    return body

log_path = f"{work}/{safe_task}.log"
_log = open(log_path, "w", encoding="utf-8")
_buf = ""
_barw = 0          # widest the bar line has been, so a redraw can clear its tail
_partial = False   # cursor is mid-line, on a bar that has not been closed yet
_bar_t0 = None
_bar_pct = -1.0
_PCT = re.compile(r"(\d+(?:\.\d+)?)%\s*$")

def bar(piece):
    # Upstream's download bars report a percentage and nothing else. Turning that
    # into elapsed and remaining seconds is the part that answers "how long".
    global _bar_t0, _bar_pct
    _m = _PCT.search(piece)
    if not _m:
        return piece
    pct, now = float(_m.group(1)), time.time()
    if _bar_t0 is None or pct < _bar_pct:
        _bar_t0 = now          # a new download started
    _bar_pct = pct
    el = now - _bar_t0
    if pct >= 99.9:
        return f"{piece}  done in {el:.0f}s"
    if el < 2 or pct <= 0:
        return piece
    return f"{piece}  {el:.0f}s elapsed, ~{el * (100 - pct) / pct:.0f}s left"

def drain():
    # Upstream writes its bar as a carriage return FOLLOWED by the bar, and never
    # clears the line, so a redraw shorter than whatever was printed before leaves
    # the tail of it behind. Each redraw is therefore padded to the widest the bar
    # has been, and a real line arriving while the cursor sits mid-bar closes that
    # line first. Reading raw bytes is what makes any of this possible - text mode
    # would have turned every carriage return into a newline and printed the bar
    # thousands of times.
    global _buf, _barw, _partial
    while True:
        _ends = [p for p in (_buf.find("\n"), _buf.find("\r")) if p >= 0]
        if not _ends:
            break
        i = min(_ends)
        piece, sep, _buf = _buf[:i], _buf[i], _buf[i + 1:]
        piece = piece.strip("\r")
        if sep == "\r":
            if not piece:
                continue
            out = bar(piece)
            _barw = max(_barw, len(out))
            sys.stdout.write("\r" + out.ljust(_barw))
            _partial = True
        else:
            if _partial:
                sys.stdout.write("\n")
                _partial, _barw = False, 0
                if not piece:
                    sys.stdout.flush()
                    continue
            _log.write(piece + "\n")
            shown = keep(piece)
            if shown is not None:
                # A finished bar arrives newline-terminated, because upstream puts
                # the carriage return before the bar rather than after it.
                if shown.startswith("[") and _PCT.search(shown):
                    shown = bar(shown)
                sys.stdout.write(shown + "\n")
        sys.stdout.flush()

t0 = time.time()
proc = subprocess.Popen(cmd, cwd=GEN, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, bufsize=0)
_decode = codecs.getincrementaldecoder("utf-8")("replace")
_fd = proc.stdout.fileno()
while True:
    _chunk = os.read(_fd, 8192)
    if not _chunk:
        break
    _buf += _decode.decode(_chunk)
    drain()
if _partial:
    sys.stdout.write("\n")
if _buf:
    _log.write(_buf + "\n")
    _tail = keep(_buf.strip("\r"))
    if _tail is not None:
        sys.stdout.write(_tail + "\n")
_log.close()
rc = proc.wait()
mins = (time.time() - t0) / 60

print("-" * 70)
if rc != 0:
    raise RuntimeError(
        f"Generation exited with code {rc} after {mins:.1f} min. The real error is the "
        f"last traceback in {log_path}, not this line. Out-of-memory is the common one "
        "on a 16 GB card: lower n_sample, shorten the binder, or crop the target "
        "harder.")

# The checkpoints and the chemical-component cache that this run downloaded for
# itself are worth keeping if step 6 was told to cache in Drive: they are the
# largest automatic download in the notebook.
if os.path.isfile("/content/.pxd_weights_cache"):
    _dw = open("/content/.pxd_weights_cache").read().strip()
    for _auto in ("ccd_cache", "checkpoint"):
        _src = f"{WEIGHTS}/{_auto}"
        if os.path.isdir(_src) and os.listdir(_src):
            print(f"copying {_auto} to Drive...")
            subprocess.run(["rsync", "-a", _src + "/", f"{_dw}/{_auto}/"])

# ---- what landed ---------------------------------------------------------
# `pipeline` collects what it generated into design_outputs/<task>/orig_designed;
# `infer` leaves the raw samples in <task>/seed_<n>/predictions. target_pred is the
# target-only prediction the Protenix filter makes to decide on templating, not a
# design, so it is not counted.
import glob
backbones = sorted(glob.glob(f"{out_dir}/design_outputs/{safe_task}/orig_designed/*.cif"))
where = f"{out_dir}/design_outputs/{safe_task}/orig_designed"
if not backbones:
    backbones = sorted(p for p in glob.glob(f"{out_dir}/**/predictions/*.cif",
                                            recursive=True)
                       if "/target_pred/" not in p)
    where = f"{out_dir}/<task>/seed_<n>/predictions"
summary = f"{out_dir}/design_outputs/{safe_task}/summary.csv"

print(f"Done in {mins:.1f} min.")
print(f"{len(backbones)} backbone structure(s) in {where}")
if not backbones:
    print("\nNothing was generated. Check the log for the real cause before re-running.")
if os.path.isfile(summary):
    import pandas as pd
    df = pd.read_csv(summary)
    print(f"\nupstream's ranked summary - {summary}")
    cols = [c for c in df.columns if "success" in c.lower()]
    for c in cols:
        try:
            print(f"  passing {c.replace('-success', '').rjust(22)} : "
                  f"{int(df[c].astype(str).str.lower().eq('true').sum())} / {len(df)}")
        except Exception:
            pass
    from IPython.display import display
    pd.set_option("display.max_columns", 200, "display.width", 250)
    display(df.head(20))
    print("\nThese have already been through ProteinMPNN and AlphaFold2 (and Protenix,")
    print("in extended mode) - `pipeline` runs PXDesignBench itself. Step 6 would")
    print("repeat that, so point it at these only to score them differently.")
    print("`orig_designed` is the ranked selection, failures included - read the table,")
    print("not the folder listing.")
else:
    print("\nNo summary.csv - `infer` mode writes backbones only. Run step 6 with")
    print("`inputs_from = PXDesign designs (step 5)` to score them.")
print(f"\nFull unfiltered log: {log_path}")
print("Step 9 copies results to Drive - /content is deleted when the runtime ends.")

In [ ]:
#@title 6. Run the evaluation
#@markdown Hands `pxdbench/run.py` (binder) or `pxdbench/run_monomer.py` (monomer)
#@markdown the arguments below and streams its output. Nothing here reimplements any
#@markdown of the pipeline — every setting is an upstream config key, named as upstream
#@markdown names it.

#@markdown ---
#@markdown ### What to evaluate
task = "binder" #@param ["binder", "monomer"]
demo_settings = True #@param {type:"boolean"}
#@markdown <small>`demo_settings` reproduces upstream's `binder_eval_demo.sh` /
#@markdown `monomer_eval_demo.sh` exactly — bundled examples, their chains, their
#@markdown sequence counts — and ignores every field below except `run_name` and
#@markdown `protenix_dtype`. **Untick it** to evaluate your own designs, or the ones
#@markdown step 5 generated.
#@markdown
#@markdown To score step 5's output: untick `demo_settings`, set `inputs_from` to
#@markdown `PXDesign designs (step 5)` and `design_task` to the task name you used,
#@markdown leave `is_mmcif` ticked, and leave `binder_chains` at `B0` — the generator
#@markdown relabels the target chains `A0, B0, …` in the order step 4 listed them and
#@markdown always puts the binder last, so with one target chain the binder is
#@markdown `B0`.</small>
run_name = "run1" #@param {type:"string"}
quiet_output = True #@param {type:"boolean"}
#@markdown <small>`quiet_output` hides the per-call bookkeeping upstream and its
#@markdown dependencies log — backend probes, featuriser notices, chain-permutation
#@markdown notes — and keeps the metrics, the progress bars and anything that looks
#@markdown like a failure. The unfiltered log is written to
#@markdown `pxd_output/&lt;run_name&gt;.log` either way, so nothing is lost.</small>
inputs_from = "bundled example" #@param ["bundled example", "PXDesign designs (step 5)", "Google Drive folder", "path on disk"]
design_task = "design1" #@param {type:"string"}
drive_subfolder = "" #@param {type:"string"}
disk_path = "" #@param {type:"string"}
is_mmcif = True #@param {type:"boolean"}
file_names = "" #@param {type:"string"}
seed = 2025 #@param {type:"integer"}
#@markdown <small>`is_mmcif` says whether the folder holds `.cif` (ticked) or `.pdb`
#@markdown (unticked) — all of them, not a mix. `file_names` restricts the run to a
#@markdown comma-separated list of names without extension; empty means the whole
#@markdown folder.</small>

#@markdown ---
#@markdown ### Chains (binder task)
binder_chains = "B0" #@param {type:"string"}
cond_chains = "" #@param {type:"string"}
orig_seqs_json = "examples/orig_seqs_test.json" #@param {type:"string"}
#@markdown <small>One binder chain only — upstream asserts that. Every other chain is
#@markdown the target unless you name `cond_chains` yourself. `orig_seqs_json` is
#@markdown optional metadata for the target: the uncropped sequence, and a
#@markdown precomputed MSA so Protenix does not have to call the MSA server. Leave it
#@markdown empty and the server is called instead, which needs a few minutes and a
#@markdown working connection.</small>

#@markdown ---
#@markdown ### Sequence design (ProteinMPNN)
num_seqs = 2 #@param {type:"integer"}
mpnn_temperature = "0.0001" #@param {type:"string"}
mpnn_binder_weights = "original" #@param ["original", "soluble"]
mpnn_monomer_model = "ca" #@param ["ca", "bb", "soluble"]
mpnn_rm_aa = "C" #@param {type:"string"}
mpnn_fix_interface = False #@param {type:"boolean"}
mpnn_bias_aa_json = "" #@param {type:"string"}
use_gt_seq = False #@param {type:"boolean"}
#@markdown <small>`num_seqs` sequences are designed per input structure. `use_gt_seq`
#@markdown skips ProteinMPNN and refolds the sequence already in the structure file.
#@markdown `mpnn_bias_aa_json` biases the amino acid composition — see
#@markdown `examples/hydrophilic_bias.json` in the repo.</small>

#@markdown ---
#@markdown ### Which filters to run (binder task)
eval_complex_af2 = True #@param {type:"boolean"}
eval_binder_monomer_af2 = True #@param {type:"boolean"}
eval_protenix_mini = True #@param {type:"boolean"}
eval_protenix_base = False #@param {type:"boolean"}
eval_diversity = False #@param {type:"boolean"}
#@markdown <small>Each one costs its own fold per sequence. `eval_protenix_base` is the
#@markdown big Protenix model and is off upstream too. `eval_diversity` is pairwise
#@markdown TM-score across the designs, on CPU.</small>

#@markdown ---
#@markdown ### AlphaFold2
af2_use_multimer = False #@param {type:"boolean"}
af2_model_ids = "0" #@param {type:"string"}
af2_use_initial_guess = True #@param {type:"boolean"}
af2_use_binder_template = True #@param {type:"boolean"}
af2_use_initial_atom_pos = False #@param {type:"boolean"}
af2_is_cyclic = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### Protenix
protenix_dtype = "fp32" #@param ["fp32", "bf16"]
ptx_n_cycle = 4 #@param {type:"integer"}
ptx_n_sample = 1 #@param {type:"integer"}
ptx_n_step = 2 #@param {type:"integer"}
ptx_use_msa = True #@param {type:"boolean"}
#@markdown <small>Keep `fp32` on a T4 — it has no bfloat16. DeepSpeed's evoformer
#@markdown attention stays off either way; it needs a build this environment does not
#@markdown have, and upstream's demos disable it too.</small>

import codecs, glob, os, re, shutil, subprocess, sys, time

ENV, REPO = "/content/pxdenv", "/content/PXDesignBench"
PY, WEIGHTS = f"{ENV}/bin/python", "/content/tool_weights"
assert os.path.isfile(PY), "run step 3 first"
assert os.path.isdir(f"{WEIGHTS}/mpnn/vanilla_model_weights"), "run step 3 first"

# Upstream's config parser zips argv into `--key value` pairs and re-splits the
# result on whitespace, so no value may contain a space and none may be empty.
# Everything therefore runs out of space-free paths under /content, and Drive is
# only ever read from and written to by this notebook, never by the subprocess.
WORK = "/content/pxd_work"
OUT_ROOT = "/content/pxd_output"
safe_run = re.sub(r"[^A-Za-z0-9._-]", "_", run_name) or "run1"
out_dir = f"{OUT_ROOT}/{safe_run}"

if demo_settings:
    # Exactly binder_eval_demo.sh / monomer_eval_demo.sh.
    if task == "binder":
        inputs_from, is_mmcif, file_names = "bundled example", True, ""
        binder_chains, cond_chains = "B0", ""
        orig_seqs_json = "examples/orig_seqs_test.json"
        num_seqs, mpnn_temperature, use_gt_seq = 2, "0.0001", False
        mpnn_binder_weights, mpnn_rm_aa, mpnn_fix_interface = "original", "C", False
        mpnn_bias_aa_json = ""
        eval_complex_af2 = eval_binder_monomer_af2 = eval_protenix_mini = True
        eval_protenix_base = eval_diversity = False
        af2_use_multimer, af2_model_ids = False, "0"
        af2_use_initial_guess = af2_use_binder_template = True
        af2_use_initial_atom_pos = af2_is_cyclic = False
        ptx_n_cycle, ptx_n_sample, ptx_n_step, ptx_use_msa = 4, 1, 2, True
    else:
        inputs_from, is_mmcif, file_names = "bundled example", False, ""
        num_seqs, mpnn_temperature, mpnn_monomer_model = 8, "0.1", "ca"
        eval_diversity, use_gt_seq, mpnn_bias_aa_json = False, False, ""
    seed = 2025
    print("Using upstream demo settings.\n")

# ---- where the designs come from -----------------------------------------
if inputs_from == "bundled example":
    data_dir = "examples/binder" if task == "binder" else "examples/monomer"
    abs_data = os.path.join(REPO, data_dir)
elif inputs_from.startswith("PXDesign designs"):
    # The generator writes mmCIF with the target chains relabelled A0, B0, ... in the
    # order step 4 listed them and the binder always last, so for one target chain the
    # binder is B0 - which is what the bundled examples are, and why the demo defaults
    # already say B0.
    _safe_task = re.sub(r"[^A-Za-z0-9._-]", "_", design_task) or "design1"
    _gen = f"/content/pxd_design/{_safe_task}/out"
    assert os.path.isdir(_gen), (
        f"no generated designs under {_gen} - run step 5 first, with this task_name")
    # Two layouts to cover: `pipeline` collects everything it generated into
    # design_outputs/<task>/orig_designed, while `infer` leaves the raw samples in
    # <task>/seed_<n>/predictions. Take the collected folder when it exists, and
    # otherwise gather the raw predictions - target_pred is excluded, as it holds the
    # target-only prediction the Protenix filter makes to decide on templating, not a
    # design.
    _orig = f"{_gen}/design_outputs/{_safe_task}/orig_designed"
    _found = sorted(glob.glob(f"{_orig}/*.cif"))
    if not _found:
        _found = sorted(p for p in glob.glob(f"{_gen}/**/predictions/*.cif",
                                             recursive=True)
                        if "/target_pred/" not in p)
    assert _found, (
        f"step 5 left no .cif designs under {_gen} - check its output for a failure")
    abs_data = f"{WORK}/{safe_run}_designs"
    shutil.rmtree(abs_data, ignore_errors=True)
    os.makedirs(abs_data)
    for _p in _found:
        shutil.copy2(_p, abs_data)
    data_dir = abs_data
    print(f"evaluating {len(_found)} generated design(s) from {_safe_task}\n")
    if not is_mmcif:
        is_mmcif = True
        print("note: the generator writes .cif, so is_mmcif has been forced on.\n")
elif inputs_from == "Google Drive folder":
    assert os.path.isfile("/content/.pxd_drive"), "Drive is not mounted - run step 2"
    src = open("/content/.pxd_drive").read().strip() + "/inputs"
    if drive_subfolder.strip():
        src = os.path.join(src, drive_subfolder.strip())
    assert os.path.isdir(src), f"no such folder in Drive: {src}"
    # Copied onto local disk first: Drive paths contain a space ("My Drive"), which
    # upstream's argument parser cannot carry, and Drive reads are slow besides.
    abs_data = f"{WORK}/{safe_run}_inputs"
    shutil.rmtree(abs_data, ignore_errors=True)
    os.makedirs(abs_data)
    ext = ".cif" if is_mmcif else ".pdb"
    picked = [f for f in sorted(os.listdir(src)) if f.lower().endswith(ext)]
    assert picked, f"no {ext} files in {src} (is `is_mmcif` set the right way?)"
    for f in picked:
        shutil.copy2(os.path.join(src, f), abs_data)
    print(f"copied {len(picked)} {ext} file(s) from Drive\n")
    data_dir = abs_data
else:
    abs_data = data_dir = disk_path.strip()
    assert abs_data and os.path.isdir(abs_data), f"not a folder: {abs_data!r}"
    assert " " not in abs_data, "path must not contain spaces"

os.makedirs(out_dir, exist_ok=True)
b = lambda v: "true" if v else "false"

# ---- the command, keyed exactly as upstream's configs are ----------------
if task == "binder":
    cmd = [PY, "-u", "pxdbench/run.py",
           "--data_dir", data_dir,
           "--dump_dir", out_dir,
           "--is_mmcif", b(is_mmcif),
           "--seed", str(seed),
           "--binder_chains", binder_chains.strip(),
           "--binder.num_seqs", str(num_seqs),
           "--binder.use_gt_seq", b(use_gt_seq),
           "--binder.eval_diversity", b(eval_diversity),
           "--binder.eval_complex", b(eval_complex_af2),
           "--binder.eval_binder_monomer", b(eval_binder_monomer_af2),
           "--binder.eval_protenix_mini", b(eval_protenix_mini),
           "--binder.eval_protenix", b(eval_protenix_base),
           "--binder.tools.mpnn.temperature", mpnn_temperature.strip(),
           "--binder.tools.mpnn.weights", mpnn_binder_weights,
           "--binder.tools.mpnn.rm_aa", mpnn_rm_aa.strip() or "C",
           "--binder.tools.mpnn.fix_interface", b(mpnn_fix_interface),
           "--binder.tools.af2.use_multimer", b(af2_use_multimer),
           "--binder.tools.af2.model_ids", af2_model_ids.replace(" ", ""),
           "--binder.tools.af2.use_initial_guess", b(af2_use_initial_guess),
           "--binder.tools.af2.use_binder_template", b(af2_use_binder_template),
           "--binder.tools.af2.use_initial_atom_pos", b(af2_use_initial_atom_pos),
           "--binder.tools.af2.is_cyclic", b(af2_is_cyclic),
           "--binder.tools.ptx_mini.load_checkpoint_dir", f"{WEIGHTS}/checkpoint",
           "--binder.tools.ptx.load_checkpoint_dir", f"{WEIGHTS}/checkpoint",
           "--binder.tools.ptx_mini.dtype", protenix_dtype,
           "--binder.tools.ptx_mini.use_deepspeed_evo_attention", "false",
           "--binder.tools.ptx_mini.N_cycle", str(ptx_n_cycle),
           "--binder.tools.ptx_mini.N_sample", str(ptx_n_sample),
           "--binder.tools.ptx_mini.N_step", str(ptx_n_step),
           "--binder.tools.ptx_mini.use_msa", b(ptx_use_msa),
           "--binder.tools.ptx.dtype", protenix_dtype,
           "--binder.tools.ptx.use_deepspeed_evo_attention", "false"]
    if cond_chains.strip():
        cmd += ["--cond_chains", cond_chains.replace(" ", "")]
    if orig_seqs_json.strip():
        cmd += ["--orig_seqs_json", orig_seqs_json.strip()]
    if mpnn_bias_aa_json.strip():
        cmd += ["--binder.tools.mpnn.bias_aa_json", mpnn_bias_aa_json.strip()]
    if file_names.strip():
        cmd += ["--file_name_list", file_names.replace(" ", "")]
else:
    cmd = [PY, "-u", "pxdbench/run_monomer.py",
           "--data_dir", data_dir,
           "--dump_dir", out_dir,
           "--is_mmcif", b(is_mmcif),
           "--seed", str(seed),
           "--monomer.num_seqs", str(num_seqs),
           "--monomer.tools.mpnn.temperature", mpnn_temperature.strip(),
           "--monomer.tools.mpnn.model_type", mpnn_monomer_model,
           "--monomer.eval_diversity", b(eval_diversity)]
    if mpnn_bias_aa_json.strip():
        cmd += ["--monomer.tools.mpnn.bias_aa_json", mpnn_bias_aa_json.strip()]

# ---- environment ---------------------------------------------------------
env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
env.update({
    # PATH first: ProteinMPNN, AlphaFold2 and Protenix each run in a subprocess that
    # upstream launches as plain `python3`, so `python3` has to mean this environment.
    "PATH": f"{ENV}/bin:" + os.environ.get("PATH", ""),
    "PYTHONPATH": REPO,
    "PYTHONNOUSERSITE": "1",
    "VIRTUAL_ENV": ENV,
    "TOOL_WEIGHTS_ROOT": WEIGHTS,
    # Where step 3 put the chemical component dictionary.
    "PROTENIX_DATA_ROOT_DIR": f"{WEIGHTS}/ccd_cache",
    "USE_DEEPSPEED_EVO_ATTENTION": "false",
    # JAX would otherwise reserve three quarters of the card up front and leave
    # nothing for the PyTorch half of the pipeline.
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
    "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.5",
})
if os.path.exists("/content/.pxd_fastln_ok"):
    env["LAYERNORM_TYPE"] = "fast_layernorm"

print("command:")
print("  " + " \\\n    ".join(cmd[1:]))
print(f"\ninput : {abs_data}")
print(f"output: {out_dir}")
print(f"fused LayerNorm: {'on' if 'LAYERNORM_TYPE' in env else 'off (torch LayerNorm)'}")
n_struct = len([f for f in os.listdir(abs_data)
                if f.lower().endswith(".cif" if is_mmcif else ".pdb")])
print(f"\n{n_struct} structure(s) x {num_seqs} sequence(s) to fold. The first")
print("evaluation also downloads the Protenix checkpoint and CCD files (~1 GB).")
print("-" * 70)

# ---- output filter -------------------------------------------------------
# Upstream and its dependencies log a line per internal call. None of it is a
# result, and on a real campaign it buries the metrics, so quiet_output drops the
# known-noise lines and strips the logger's timestamp-and-path prefix from the
# rest. Anything unrecognised is printed, which keeps errors visible.
_NOISE = re.compile(
    r"PROTENIX_DATA_ROOT_DIR not set"
    r"|xla_bridge"
    r"|Featurizing "
    r"|Loaded constraint feature"
    r"|use_msa is \w+ for sequence"
    r"|No need to perform chain permutation"
    r"|Rebuilt and renumbered PDB saved"
    r"|WARNING: 'model_\d_ptm' not found"
    r"|Run subprocess success: 0"
    r"|Distributed environment:|LOCAL_RANK:|Finished init ENV"
    r"|train scheduler |Sampled key:|Loading protenix filter model"
    r"|^\{[\d:, ]+\} [\d.]+$"
)
_STAMP = re.compile(r"^\d{4}-\d\d-\d\d \d\d:\d\d:\d\d,\d+ \[[^\]]+\] \w+ [\w.]+: ")

def keep(line):
    # The line to print, or None to drop it.
    if not quiet_output:
        return line
    body = _STAMP.sub("", line).rstrip()
    # Matched against both, because the prefix carries the logger name and some of
    # the noise is only identifiable there.
    if not body.strip() or _NOISE.search(line) or _NOISE.search(body):
        return None
    if body.startswith("Missing columns"):
        _f = re.search(r"for filter '([^']+)'", body)
        return f"note: filter '{_f.group(1)}' skipped - its model was not run"
    return body

log_path = f"{OUT_ROOT}/{safe_run}.log"
os.makedirs(OUT_ROOT, exist_ok=True)
_log = open(log_path, "w", encoding="utf-8")
_buf = ""
_barw = 0          # widest the bar line has been, so a redraw can clear its tail
_partial = False   # cursor is mid-line, on a bar that has not been closed yet
_bar_t0 = None
_bar_pct = -1.0
_PCT = re.compile(r"(\d+(?:\.\d+)?)%\s*$")

def bar(piece):
    # Upstream's download bars report a percentage and nothing else. Turning that
    # into elapsed and remaining seconds is the part that answers "how long".
    global _bar_t0, _bar_pct
    _m = _PCT.search(piece)
    if not _m:
        return piece
    pct, now = float(_m.group(1)), time.time()
    if _bar_t0 is None or pct < _bar_pct:
        _bar_t0 = now          # a new download started
    _bar_pct = pct
    el = now - _bar_t0
    if pct >= 99.9:
        return f"{piece}  done in {el:.0f}s"
    if el < 2 or pct <= 0:
        return piece
    return f"{piece}  {el:.0f}s elapsed, ~{el * (100 - pct) / pct:.0f}s left"

def drain():
    # Upstream writes its bar as a carriage return FOLLOWED by the bar, and never
    # clears the line, so a redraw shorter than whatever was printed before leaves
    # the tail of it behind. Each redraw is therefore padded to the widest the bar
    # has been, and a real line arriving while the cursor sits mid-bar closes that
    # line first. Reading raw bytes is what makes any of this possible - text mode
    # would have turned every carriage return into a newline and printed the bar
    # thousands of times.
    global _buf, _barw, _partial
    while True:
        _ends = [p for p in (_buf.find("\n"), _buf.find("\r")) if p >= 0]
        if not _ends:
            break
        i = min(_ends)
        piece, sep, _buf = _buf[:i], _buf[i], _buf[i + 1:]
        piece = piece.strip("\r")
        if sep == "\r":
            if not piece:
                continue
            out = bar(piece)
            _barw = max(_barw, len(out))
            sys.stdout.write("\r" + out.ljust(_barw))
            _partial = True
        else:
            if _partial:
                sys.stdout.write("\n")
                _partial, _barw = False, 0
                if not piece:
                    sys.stdout.flush()
                    continue
            _log.write(piece + "\n")
            shown = keep(piece)
            if shown is not None:
                # A finished bar arrives newline-terminated, because upstream puts
                # the carriage return before the bar rather than after it.
                if shown.startswith("[") and _PCT.search(shown):
                    shown = bar(shown)
                sys.stdout.write(shown + "\n")
        sys.stdout.flush()

t0 = time.time()
proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, bufsize=0)

_decode = codecs.getincrementaldecoder("utf-8")("replace")
_fd = proc.stdout.fileno()
while True:
    _chunk = os.read(_fd, 8192)
    if not _chunk:
        break
    _buf += _decode.decode(_chunk)
    drain()
if _partial:
    sys.stdout.write("\n")
if _buf:
    _log.write(_buf + "\n")
    _tail = keep(_buf.strip("\r"))
    if _tail is not None:
        sys.stdout.write(_tail + "\n")
    _buf = ""
_log.close()
rc = proc.wait()
mins = (time.time() - t0) / 60

print("-" * 70)
if rc != 0:
    raise RuntimeError(
        f"Evaluation exited with code {rc} after {mins:.1f} min. The real error is "
        f"the last traceback in {log_path}, not this line. Out-of-memory on a "
        "T4 is the common one: turn off a filter, or lower num_seqs.")
if os.path.isfile("/content/.pxd_weights_cache"):
    _dw = open("/content/.pxd_weights_cache").read().strip()
    for _auto in ("ccd_cache", "checkpoint"):
        _src = f"{WEIGHTS}/{_auto}"
        if os.path.isdir(_src) and os.listdir(_src):
            print(f"copying {_auto} to Drive...")
            subprocess.run(["rsync", "-a", _src + "/", f"{_dw}/{_auto}/"])

print(f"Done in {mins:.1f} min. Results in {out_dir}")
print(f"Full unfiltered log: {log_path}")
print("Run step 9 before closing the tab - /content is deleted when the runtime ends.")

In [ ]:
#@title 7. Results
#@markdown Reads the two files upstream writes — `summary_output.json`, the averages
#@markdown and pass counts over the whole run, and `sample_level_output.csv`, one row
#@markdown per designed sequence.
run_name = "run1" #@param {type:"string"}
sort_by = "" #@param {type:"string"}
#@markdown <small>`sort_by` is any column of the CSV, best first — try `i_pTM` or
#@markdown `scRMSD`. Empty keeps upstream's order.</small>

import json, os, re, glob
import pandas as pd
from IPython.display import display

safe_run = re.sub(r"[^A-Za-z0-9._-]", "_", run_name) or "run1"
out_dir = f"/content/pxd_output/{safe_run}"
assert os.path.isdir(out_dir), f"no such run: {out_dir} - check run_name"

summaries = sorted(glob.glob(f"{out_dir}/**/summary_output.json", recursive=True))
samples = sorted(glob.glob(f"{out_dir}/**/sample_level_output.csv", recursive=True))
assert summaries or samples, f"{out_dir} holds no results - did step 6 finish?"

pd.set_option("display.max_columns", 200, "display.width", 250)

for s in summaries:
    with open(s) as f:
        summary = json.load(f)
    print("=" * 70)
    print(os.path.relpath(s, out_dir))
    print("=" * 70)
    # Pass counts first: they are the numbers the filters exist to produce.
    passes = {k: v for k, v in summary.items() if k.endswith("_success.count")}
    if passes:
        print("\ndesigns passing each filter")
        for k, v in sorted(passes.items()):
            print(f"  {k.replace('_success.count', '').rjust(12)} : {v}")
    print("\nsummary")
    for k, v in summary.items():
        if k in passes:
            continue
        print(f"  {k.rjust(40)} : {v if not isinstance(v, float) else round(v, 4)}")

for c in samples:
    df = pd.read_csv(c)
    print("\n" + "=" * 70)
    print(f"{os.path.relpath(c, out_dir)}  -  {len(df)} rows")
    print("=" * 70)
    if sort_by.strip():
        col = sort_by.strip()
        if col in df.columns:
            ascending = any(t in col.lower() for t in ("rmsd", "pae", "loss"))
            df = df.sort_values(col, ascending=ascending)
        else:
            print(f"(no column {col!r}; columns are: {', '.join(df.columns)})\n")
    display(df)

In [ ]:
#@title 8. View a prediction
#@markdown Overlays a refolded structure on the design it came from. Grey is the
#@markdown design that went in; coloured is what the folding model predicted from the
#@markdown designed sequence alone. The closer they sit, the better that design scored.
run_name = "run1" #@param {type:"string"}
prediction = 0 #@param {type:"integer"}
#@markdown <small>`prediction` indexes the list this cell prints. Run it once to see
#@markdown the list, then set the number.</small>

import glob, os, re

try:
    import py3Dmol
except ImportError:
    !pip install -q py3Dmol
    import py3Dmol

safe_run = re.sub(r"[^A-Za-z0-9._-]", "_", run_name) or "run1"
out_dir = f"/content/pxd_output/{safe_run}"
assert os.path.isdir(out_dir), f"no such run: {out_dir} - check run_name"

preds = sorted(p for p in glob.glob(f"{out_dir}/**/*.pdb", recursive=True)
               if "converted_pdbs" not in p)
assert preds, f"no predicted structures under {out_dir}"

print(f"{len(preds)} prediction(s):")
for i, p in enumerate(preds):
    print(f"  [{i}] {os.path.relpath(p, out_dir)}")

idx = max(0, min(prediction, len(preds) - 1))
pred_path = preds[idx]
print(f"\nshowing [{idx}] {os.path.relpath(pred_path, out_dir)}")

# The design the prediction is named after: strip the _seqN / _modelN suffixes
# upstream appends, then look for that name among the inputs.
stem = os.path.basename(pred_path)[:-4]
stem = re.sub(r"_model\d+$", "", stem)
stem = re.sub(r"_seed_\d+.*$", "", stem)
base = re.sub(r"_seq\d+$", "", stem)
design = None
for cand in glob.glob(f"/content/pxd_work/**/{base}.pdb", recursive=True) + \
            glob.glob(f"/content/PXDesignBench/examples/**/{base}.pdb", recursive=True) + \
            glob.glob(f"{out_dir}/**/converted_pdbs/{base}.pdb", recursive=True):
    design = cand
    break

view = py3Dmol.view(width=900, height=600)
view.addModel(open(pred_path).read(), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"colorscheme": "chain"}})
if design:
    view.addModel(open(design).read(), "pdb")
    view.setStyle({"model": 1}, {"cartoon": {"color": "lightgrey", "opacity": 0.65}})
    print(f"design overlay: {design}")
else:
    print(f"(no input structure found for {base!r} - showing the prediction alone)")
view.zoomTo()
view.show()

In [ ]:
#@title 9. Save results to Drive
#@markdown Colab wipes `/content` when the runtime ends. Run this before closing the
#@markdown tab.
run_name = "run1" #@param {type:"string"}
design_task = "" #@param {type:"string"}
include_structures = True #@param {type:"boolean"}
#@markdown <small>`run_name` is the evaluation from step 6; leave it empty if you only
#@markdown generated. `design_task` is step 5's task name — set it to copy the
#@markdown generated backbones and their ranked `summary.csv` as well. Untick
#@markdown `include_structures` to copy only the CSV and JSON summaries and leave the
#@markdown structures behind, which is much smaller if Drive is tight.</small>

import os, re, subprocess
from datetime import datetime

assert os.path.isfile("/content/.pxd_drive"), "Drive is not mounted - run step 2"
drive = open("/content/.pxd_drive").read().strip()
stamp = datetime.now().strftime("%Y%m%d_%H%M")

jobs = []
if run_name.strip():
    safe_run = re.sub(r"[^A-Za-z0-9._-]", "_", run_name)
    src = f"/content/pxd_output/{safe_run}"
    assert os.path.isdir(src), f"no such evaluation: {src} - check run_name"
    jobs.append((src, f"{drive}/outputs/{safe_run}_{stamp}"))
if design_task.strip():
    safe_task = re.sub(r"[^A-Za-z0-9._-]", "_", design_task)
    src = f"/content/pxd_design/{safe_task}"
    assert os.path.isdir(src), f"no such design task: {src} - check design_task"
    jobs.append((src, f"{drive}/designs/{safe_task}_{stamp}"))
assert jobs, "nothing to copy - set run_name, design_task, or both"

for src, dest in jobs:
    os.makedirs(dest, exist_ok=True)
    args = ["rsync", "-a", "--info=stats1"]
    if not include_structures:
        args += ["--include=*/", "--include=*.csv", "--include=*.json",
                 "--include=*.yaml", "--include=*.log", "--exclude=*"]
    args += [src + "/", dest + "/"]
    subprocess.run(args, check=True)
    total = sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(dest) for f in fs)
    n = sum(len(fs) for _, _, fs in os.walk(dest))
    print(f"copied {n} file(s), {total / 1e6:.1f} MB")
    print(f"to: {dest.replace('/content/drive/MyDrive', 'My Drive')}\n")

---
## The generated designs (step 5)

`pipeline` mode writes its own ranked table, separate from step 6's:
`out/design_outputs/<task_name>/summary.csv`, one row per design, with upstream's
four filters as `True`/`False` columns. Its thresholds, from upstream's README:

| Filter | Confidence | Geometry |
|---|---|---|
| `AF2-IG-easy` | ipAE < 10.85, ipTM > 0.5, pLDDT > 0.8 | bound/unbound RMSD < 3.5 A |
| `AF2-IG` | ipAE < 7.0, pLDDT > 0.9 | binder RMSD < 1.5 A |
| `Protenix-basic` | binder ipTM > 0.8, binder pTM > 0.8 | complex RMSD < 2.5 A |
| `Protenix` | binder ipTM > 0.85, binder pTM > 0.88 | complex RMSD < 2.5 A |

`AF2-IG-easy` is BindCraft's threshold set; the others are upstream's own.

Alongside `summary.csv` are the structures, one file per row named `rank_<n>.cif`:

- `orig_designed/` — the designed backbone as PXDesign-d drew it. This is the
  ranked *selection*, not every sample: upstream returns up to
  `max_success_return` successes and pads the list to `min_total_return` with
  failures, and the CLI sets both to `n_sample`. Designs that failed every filter
  are in there too, which is why `summary.csv` is the thing to read rather than
  the folder listing.
- `passing-AF2-IG-easy/` — for rows that cleared that filter, AF2's own docked
  pose rather than the original backbone.
- `passing-Protenix-basic/` — the same, from Protenix.

`summary.csv`'s `chosen_struct_path` names which of the three upstream considers
the best structure for that row.

`infer` mode skips all of this: it writes raw samples to
`<task>/seed_<n>/predictions/` and no table. Score them with step 6, which gives
you every knob instead of a preset.

Upstream's guidance on picking designs to actually order: generate thousands across
several seeds and binder lengths, aiming for 10-100 that pass both the Protenix and
AF2-IG filters; cluster what survives by structure; then rank by Protenix `ipTM`
within each cluster and take the top of each. On a T4 that scale means many sessions,
which is the honest reason upstream recommends their web server.

## What the numbers mean

One row per designed sequence in `sample_level_output.csv`, averaged into
`summary_output.json`. The ones the filters are built on:

| Column | Reading it |
|---|---|
| `pLDDT` | AlphaFold2's confidence in the predicted complex. Higher is better. |
| `i_pTM` | Confidence in the *interface* specifically. Higher is better. |
| `i_pAE` | Expected error across the interface. **Lower** is better. |
| `af2_binder_pred_design_rmsd` | How far AF2's binder drifts from the designed backbone. **Lower** is better. |
| `bound_unbound_RMSD` | How much the binder changes shape when the target is taken away. **Lower** means it holds its fold on its own. |
| `ptx_mini_iptm_binder` | The same interface question asked of Protenix instead of AF2. Higher is better. |
| `scRMSD` / `scTM` | Monomer self-consistency: refold the designed sequence with ESMFold and compare. Lower RMSD / higher TM is better. |
| `alpha`, `beta`, `loop`, `Rg` | Secondary structure fractions and radius of gyration — shape, not quality. |
| `*_success` | 1 if that row passed every threshold in that filter, 0 if not. `af2_easy`, `af2_opt`, `ptx_mini` and the rest are defined in `pxdbench/pxd_configs/eval.py`. |

Thresholds are upstream's and are not changed here. To see or alter them, open
`/content/PXDesignBench/pxdbench/pxd_configs/eval.py` in the Files pane.

## If something fails

**Step 3 raises.** Read which check failed. NumPy pulled back to 2.x or JAX landing
on CPU both mean a dependency resolved differently than the pins intend; the fix is
*Runtime > Disconnect and delete runtime* and a fresh start, not re-running the cell
on top of a half-built environment.

**`CUDA out of memory` during step 5 or 6.** A T4 has 16 GB and this pipeline keeps
AlphaFold2 (JAX) and Protenix (PyTorch) alive on the same card. Untick
`eval_protenix_mini`, or lower `num_seqs`, or evaluate fewer structures per run with
`file_names`. A long target chain costs more than a long binder.

**It sits silent for a long time on the first run.** Expected twice: the PXDesign
and Protenix checkpoints are downloaded when first needed (~1 GB and up), and
AlphaFold2 compiles its JAX graph for each new sequence length. Later designs of the same length reuse the compilation.

**Step 5 raises on `bf16`.** A T4 has no bfloat16 unit. Set `dtype` to `fp32`; step 1
prints whether your card has one.

**A hotspot reads `NOT FOUND` in step 4.** The number is an `auth_seq_id` or is past
the end of the chain. PXDesign uses mmCIF `label_seq_id` — in
[Mol\*](https://molstar.org/viewer/) that is *Sequence ID*, not *Auth ID*.

**The MSA step hangs or errors.** With `orig_seqs_json` empty, Protenix asks a public
MSA server for the target's alignment, which is rate-limited and sometimes down. For
repeated work on one target, compute the MSA once and point `orig_seqs_json` at a
file naming it under `msa.precomputed_msa_dir` — the bundled
`examples/orig_seqs_test.json` is the pattern to copy.

**`Missing tool weight`.** Step 3 did not finish, or you set `install_for` to a task
you then did not run — the binder pipeline needs AlphaFold2, the monomer pipeline
needs ESMFold. Re-run step 3 with the right `install_for`.

**Nothing passes any filter.** That is a result, not a failure: the thresholds are
the ones from the PXDesign paper and real designs fail them routinely. `af2_easy` is
the lenient set; start there.

## Credit

The generator is [PXDesign](https://github.com/bytedance/PXDesign) and the evaluation
is [PXDesignBench](https://github.com/bytedance/PXDesignBench), both by ByteDance,
installed from source and run unmodified. They stand on
[Protenix](https://github.com/bytedance/Protenix),
[ProteinMPNN](https://github.com/dauparas/ProteinMPNN),
[AlphaFold2](https://github.com/google-deepmind/alphafold) via
[ColabDesign](https://github.com/sokrypton/ColabDesign), and
[ESMFold](https://huggingface.co/facebook/esmfold_v1). Cite those, not this notebook —
the citations are listed in upstream's README.

The AlphaFold2 parameters are released under CC BY-NC 4.0: non-commercial use only.
That restriction travels with the weights, whatever the code around them is licensed as.